# Week 1 — DINOv2 on DESI Legacy Survey DR10 galaxies

**Project:** Self-Supervised Vision Transformers for Unsupervised Object and Part Discovery

This notebook covers the Week 1 groundwork:

| # | Section | What you get |
|---|---------|--------------|
| 1 | Environment setup | packages, GPU check, config |
| 2 | Load DESI dataset | streamed DR10 cutouts (g, r, i, z · 160×160) cached to disk |
| 3 | Dataset exploration | fields, shapes, units, catalog statistics |
| 4 | Visualize g/r/i/z | per-band images, colour composites, provided masks |
| 5 | Load pretrained DINOv2 | backbone from `torch.hub` |
| 6 | Preprocess image | 4-band flux → RGB → normalised tensor |
| 7 | Run inference | `forward_features` on one image and a batch |
| 8 | Extract CLS token | global embedding |
| 9 | Extract patch tokens | local (per-patch) embeddings |
| 10 | Inspect tensor dimensions | shapes vs. input resolution and model size |
| 11 | Visualize patch grid | grid overlay, token norms, patch PCA, attention preview |
| 12 | Initial embedding exploration | PCA / t-SNE, nearest neighbours, optional Galaxy Zoo labels |

**Before running:** `Runtime → Change runtime type → T4 GPU`. Everything runs on the free tier.

## 1. Environment setup

In [ ]:
# Colab ships torch, torchvision, sklearn, matplotlib, pandas and astropy.
# We only need a recent `datasets` for streaming the Multimodal Universe data.
!pip -q install -U "datasets>=2.19" "huggingface_hub>=0.23"

In [ ]:
import os, time, math, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU found — everything still runs, just slower. "
          "In Colab: Runtime → Change runtime type → T4 GPU.")

plt.rcParams.update({"figure.dpi": 110, "image.origin": "lower",
                     "axes.titlesize": 9, "font.size": 9})

In [ ]:
# ---- Configuration: change things here, not deep inside the notebook ----
CFG = dict(
    dataset="MultimodalUniverse/legacysurvey",  # DR10 cutouts, 4 bands, 160x160
    n_samples=1000,          # galaxies to stream for Week 1 (~1 MB each on the wire)
    shuffle_buffer=500,      # streaming shuffle: mixes shards so we don't get one sky patch
    model_name="dinov2_vits14",  # try later: dinov2_vitb14, dinov2_vits14_reg
    img_size=224,            # must be a multiple of 14 -> 224/14 = 16x16 patch grid
    batch_size=64,
    cache_dir="/content/week1_cache" if os.path.exists("/content") else "./week1_cache",
)
os.makedirs(CFG["cache_dir"], exist_ok=True)

# Optional: keep the cache on Google Drive so it survives runtime resets.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CFG["cache_dir"] = "/content/drive/MyDrive/dino_desi/week1_cache"
    os.makedirs(CFG["cache_dir"], exist_ok=True)

CACHE_FILE = os.path.join(CFG["cache_dir"], f"legacysurvey_{CFG['n_samples']}.npz")
print(json.dumps(CFG, indent=2))

## 2. Load DESI dataset

The DR10 cutouts live on Hugging Face as part of the **Multimodal Universe** collection
(`MultimodalUniverse/legacysurvey`, ~90.6k galaxies, ~64 GB). Each row contains:

* `image` — one entry **per band** (`des-g, des-r, des-i, des-z`), each with `flux` (160×160, float32),
  `mask`, `ivar` (inverse variance), `psf_fwhm` and pixel `scale`
* `rgb`, `object_mask`, `blobmodel` — pre-rendered 160×160 images
* catalog columns — `FLUX_G/R/I/Z`, WISE `FLUX_W1–W4`, `SHAPE_R` (half-light radius), `SHAPE_E1/E2`, `EBV`, `object_id`

We **stream** a random subset instead of downloading 64 GB, then cache it as a `.npz` so re-runs are instant.
Streaming shuffles shard order as well as rows, so the sample isn't all from one patch of sky.

In [ ]:
BANDS = ["g", "r", "i", "z"]
SCALAR_COLS = ["EBV", "FLUX_G", "FLUX_R", "FLUX_I", "FLUX_Z",
               "FLUX_W1", "FLUX_W2", "FLUX_W3", "FLUX_W4",
               "SHAPE_R", "SHAPE_E1", "SHAPE_E2"]

def _band_letter(name):
    # 'des-g' -> 'g'
    return str(name).strip().split("-")[-1].lower()

def parse_image_field(img):
    # The per-band Sequence arrives either as a dict of lists or a list of dicts,
    # depending on the `datasets` version. Normalise to dict of lists.
    if isinstance(img, (list, tuple)):
        img = {k: [d[k] for d in img] for k in img[0].keys()}
    letters = [_band_letter(b) for b in img["band"]]
    order = [letters.index(b) for b in BANDS]          # force g, r, i, z order
    flux = np.stack([np.asarray(img["flux"][j], dtype=np.float32) for j in order])
    mask = np.stack([np.asarray(img["mask"][j], dtype=bool) for j in order])
    psf = np.asarray([img["psf_fwhm"][j] for j in order], dtype=np.float32)
    scale = np.asarray([img["scale"][j] for j in order], dtype=np.float32)
    return flux, mask, psf, scale

def _clean_id(x):
    s = x.decode() if isinstance(x, bytes) else str(x)
    return s[2:-1] if s.startswith("b'") and s.endswith("'") else s

def stream_legacysurvey(n, seed=SEED, buffer_size=500):
    from datasets import load_dataset
    ds = load_dataset(CFG["dataset"], split="train", streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=buffer_size)
    flux, mask, psf, scale, rgb, objmask, rows = [], [], [], [], [], [], []
    t0 = time.time()
    for i, ex in enumerate(ds):
        if i >= n:
            break
        f, m, p, s = parse_image_field(ex["image"])
        flux.append(f); mask.append(m); psf.append(p); scale.append(s)
        rgb.append(np.asarray(ex["rgb"].convert("RGB"), dtype=np.uint8))
        objmask.append(np.asarray(ex["object_mask"].convert("L"), dtype=np.uint8))
        row = {c: float(ex[c]) for c in SCALAR_COLS}
        row["object_id"] = _clean_id(ex["object_id"])
        rows.append(row)
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{n} galaxies  ({time.time()-t0:.0f}s)")
    return dict(flux=np.stack(flux), mask=np.stack(mask), psf_fwhm=np.stack(psf),
                pixscale=np.stack(scale), rgb=np.stack(rgb), object_mask=np.stack(objmask),
                catalog=pd.DataFrame(rows))

if os.path.exists(CACHE_FILE):
    print("Loading cached sample:", CACHE_FILE)
    z = np.load(CACHE_FILE, allow_pickle=False)
    data = {k: z[k] for k in ["flux", "mask", "psf_fwhm", "pixscale", "rgb", "object_mask"]}
    data["catalog"] = pd.read_csv(CACHE_FILE.replace(".npz", "_catalog.csv"))
else:
    print(f"Streaming {CFG['n_samples']} galaxies from {CFG['dataset']} …")
    data = stream_legacysurvey(CFG["n_samples"], buffer_size=CFG["shuffle_buffer"])
    np.savez(CACHE_FILE, **{k: v for k, v in data.items() if k != "catalog"})
    data["catalog"].to_csv(CACHE_FILE.replace(".npz", "_catalog.csv"), index=False)
    print("Cached to", CACHE_FILE)

flux, cat = data["flux"], data["catalog"]
N = len(flux)
print(f"\nflux array: {flux.shape}  ({flux.nbytes/1e6:.0f} MB)  — (N, bands[g,r,i,z], H, W)")

## 3. Dataset exploration

Units: Legacy Survey fluxes are in **nanomaggies**, so an AB magnitude is `m = 22.5 − 2.5·log10(flux)`.
Pixel values in the cutouts are nanomaggies per pixel, already sky-subtracted (so they can be negative).

In [ ]:
print("Array fields:")
for k in ["flux", "mask", "psf_fwhm", "pixscale", "rgb", "object_mask"]:
    v = data[k]
    print(f"  {k:12s} shape={str(v.shape):22s} dtype={v.dtype}")

print("\nPixel scale per band (arcsec/px):", np.unique(np.round(data["pixscale"], 3)))
print(f"Cutout size: {flux.shape[-1]} px = {flux.shape[-1]*float(np.median(data['pixscale'])):.1f} arcsec")
print("Median PSF FWHM per band (g, r, i, z):", np.round(np.median(data["psf_fwhm"], 0), 2))

# Per-band coverage: is the band actually observed? (DR10 i-band does not cover the full footprint)
has_signal = np.abs(flux).reshape(N, 4, -1).max(-1) > 0
print("\nFraction of cutouts with non-zero data per band:")
for b, frac in zip(BANDS, has_signal.mean(0)):
    print(f"  {b}: {frac:.1%}")
print("\nFraction of pixels with mask=True per band:",
      dict(zip(BANDS, np.round(data["mask"].mean(axis=(0, 2, 3)), 4))))
print("Non-finite pixels:", int((~np.isfinite(flux)).sum()))

In [ ]:
# Derived physical quantities from the catalog — used later to colour embeddings.
def nmgy_to_mag(f):
    f = np.asarray(f, dtype=float)
    return np.where(f > 0, 22.5 - 2.5 * np.log10(np.clip(f, 1e-9, None)), np.nan)

cat["mag_g"], cat["mag_r"], cat["mag_z"] = (nmgy_to_mag(cat[c]) for c in ["FLUX_G", "FLUX_R", "FLUX_Z"])
cat["g_minus_r"] = cat["mag_g"] - cat["mag_r"]
cat["r_minus_z"] = cat["mag_r"] - cat["mag_z"]
cat["log_size"] = np.log10(cat["SHAPE_R"].clip(lower=1e-3))      # half-light radius, arcsec
cat["ellipticity"] = np.hypot(cat["SHAPE_E1"], cat["SHAPE_E2"])

display(cat[["object_id", "mag_r", "g_minus_r", "r_minus_z", "SHAPE_R", "ellipticity", "EBV"]].describe().round(3))

fig, ax = plt.subplots(1, 5, figsize=(16, 2.8))
for a, col, lab in zip(ax, ["mag_r", "g_minus_r", "r_minus_z", "log_size", "ellipticity"],
                       ["r mag", "g − r", "r − z", "log10 half-light radius [″]", "ellipticity"]):
    v = cat[col].replace([np.inf, -np.inf], np.nan).dropna()
    a.hist(v.clip(*np.nanpercentile(v, [0.5, 99.5])), bins=40, color="#4C72B0")
    a.set_xlabel(lab)
ax[0].set_ylabel("count")
plt.suptitle("Catalog properties of the sample", y=1.03); plt.tight_layout(); plt.show()

In [ ]:
# Pixel statistics per band: noise level (robust sigma) and dynamic range.
def robust_sigma(x):
    med = np.median(x)
    return 1.4826 * np.median(np.abs(x - med))

stats = []
for bi, b in enumerate(BANDS):
    sub = flux[:200, bi]
    border = np.concatenate([sub[:, :10].ravel(), sub[:, -10:].ravel()])   # mostly sky
    centre = sub[:, 75:85, 75:85].reshape(len(sub), -1).max(1)
    stats.append(dict(band=b, sky_sigma=robust_sigma(border), centre_peak_median=np.median(centre),
                      min=float(sub.min()), max=float(sub.max())))
stats = pd.DataFrame(stats)
stats["peak/sigma"] = stats["centre_peak_median"] / stats["sky_sigma"]
display(stats.round(5))
print("Dynamic range spans several orders of magnitude -> a linear map to [0,1] would hide "
      "everything except the galaxy core. That's why we use an asinh stretch below.")

## 4. Visualize g/r/i/z

Each band is shown with its own **asinh stretch** (linear near the noise level, logarithmic for bright cores).
Then two colour composites: our own (z→R, r→G, g→B, following the Legacy Survey viewer's `sdss_rgb`
convention) and the `rgb` image shipped with the dataset, followed by the provided `object_mask`.

In [ ]:
def asinh_band(img, soft_sigmas=2.0, pct=99.7):
    # Per-band display stretch: softening = a few sky sigmas, then scale to [0, 1].
    s = robust_sigma(img) * soft_sigmas + 1e-12
    y = np.arcsinh(img / s)
    lo, hi = np.percentile(y, 1), np.percentile(y, pct)
    return np.clip((y - lo) / (hi - lo + 1e-12), 0, 1)

def sdss_rgb(flux4, bands=("g", "r", "z"), m=0.03, Q=20.0,
             rgbscales={"g": (2, 6.0), "r": (1, 3.4), "i": (0, 3.0), "z": (0, 2.2)}):
    # Legacy-Survey-style colour composite. flux4: (4, H, W) in g, r, i, z order.
    # m: black-level offset, Q: asinh softening. Returns float32 (H, W, 3) in [0, 1].
    imgs = [flux4[BANDS.index(b)] for b in bands]
    I = np.zeros_like(imgs[0])
    for img, b in zip(imgs, bands):
        I = I + np.maximum(0, img * rgbscales[b][1] + m)
    I = I / len(bands)
    fI = np.arcsinh(Q * I) / np.sqrt(Q)
    I = I + (I == 0) * 1e-6
    rgb = np.zeros(I.shape + (3,), np.float32)
    for img, b in zip(imgs, bands):
        plane, scale = rgbscales[b]
        rgb[..., plane] = (img * scale + m) * fI / I
    return np.clip(rgb, 0, 1)

# Pick a few galaxies spanning the size distribution so we see small and extended ones.
order = np.argsort(cat["SHAPE_R"].values)
show_idx = order[np.linspace(int(0.1 * N), N - 1, 6).astype(int)]

ncol = 4 + 3
fig, axes = plt.subplots(len(show_idx), ncol, figsize=(ncol * 1.9, len(show_idx) * 2.0))
for r, i in enumerate(show_idx):
    for c, b in enumerate(BANDS):
        axes[r, c].imshow(asinh_band(flux[i, c]), cmap="gray")
    axes[r, 4].imshow(sdss_rgb(flux[i]))
    axes[r, 5].imshow(data["rgb"][i], origin="upper")      # PNG/JPEG rows go top-down
    axes[r, 6].imshow(data["object_mask"][i], cmap="viridis", origin="upper")
    axes[r, 0].set_ylabel(f"#{i}\nR_e={cat['SHAPE_R'].iloc[i]:.1f}″", fontsize=8)
for c, t in enumerate([*BANDS, "our g,r,z RGB", "provided rgb", "object_mask"]):
    axes[0, c].set_title(t)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print("Check: if 'our RGB' looks vertically flipped relative to 'provided rgb', "
      "set plt.rcParams['image.origin'] = 'upper' and re-run. Orientation doesn't matter for the model.")

In [ ]:
# Per-band light profiles of one extended galaxy: how much structure does each band carry?
i = show_idx[-1]
yy, xx = np.indices(flux.shape[-2:])
rr = np.hypot(yy - 79.5, xx - 79.5).astype(int)
plt.figure(figsize=(5, 3.2))
for c, b in enumerate(BANDS):
    prof = np.bincount(rr.ravel(), flux[i, c].ravel()) / np.bincount(rr.ravel())
    plt.plot(prof[:80], label=b)
plt.yscale("symlog", linthresh=1e-3); plt.xlabel("radius [px]"); plt.ylabel("mean flux [nmgy/px]")
plt.title(f"Radial profile, galaxy #{i}"); plt.legend(); plt.tight_layout(); plt.show()

## 5. Load pretrained DINOv2

`torch.hub` pulls the official model code from GitHub and the weights from Meta's CDN.

| model | params | embed dim | heads | patch |
|---|---|---|---|---|
| `dinov2_vits14` | 21 M | 384 | 6 | 14 |
| `dinov2_vitb14` | 86 M | 768 | 12 | 14 |
| `dinov2_vits14_reg` | 21 M | 384 | 6 | 14 (+4 register tokens) |

Start with ViT-S/14; it's fast on a T4 and enough for exploration. The `_reg` variants remove the
high-norm "artifact" tokens that appear in plain DINOv2 attention maps — worth comparing in Week 2.

In [ ]:
model = torch.hub.load("facebookresearch/dinov2", CFG["model_name"])
model = model.to(device).eval()
for p in model.parameters():
    p.requires_grad_(False)

PATCH = model.patch_size
EMBED_DIM = model.embed_dim
N_REG = getattr(model, "num_register_tokens", 0)

def _all_blocks(m):
    out = []
    for b in m.blocks:
        if isinstance(b, torch.nn.ModuleList):           # BlockChunk when block_chunks > 0
            out += [x for x in b if not isinstance(x, torch.nn.Identity)]
        else:
            out.append(b)
    return out

BLOCKS = _all_blocks(model)
N_HEADS = BLOCKS[-1].attn.num_heads
print(f"{CFG['model_name']}: {sum(p.numel() for p in model.parameters())/1e6:.1f} M params | "
      f"patch={PATCH} | dim={EMBED_DIM} | depth={len(BLOCKS)} | heads={N_HEADS} | registers={N_REG}")

## 6. Preprocess image

DINOv2 was trained on natural RGB photos, so for this baseline we:

1. **4 bands → 3 channels**: build the g, r, z colour composite (`sdss_rgb`). The i-band is dropped here
   (it's also missing for part of the DR10 footprint). Using all 4 bands is a later experiment
   (e.g. a 4-channel patch embedding).
2. **Resize** 160 → `img_size` (224 by default → a 16×16 patch grid with patch size 14).
3. **Normalise** with ImageNet mean/std, which is what DINOv2 expects.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def flux_to_rgb_batch(flux_batch):
    # (B, 4, H, W) flux -> (B, H, W, 3) float RGB in [0, 1]
    return np.stack([sdss_rgb(f) for f in flux_batch])

def preprocess(rgb_batch, size=CFG["img_size"]):
    # (B, H, W, 3) in [0,1] -> normalised tensor (B, 3, size, size)
    assert size % PATCH == 0, f"img_size must be a multiple of {PATCH}"
    x = torch.from_numpy(np.ascontiguousarray(rgb_batch)).permute(0, 3, 1, 2).float()
    if x.shape[-1] != size:
        x = F.interpolate(x, size=(size, size), mode="bicubic", align_corners=False, antialias=True)
    x = x.clamp(0, 1)
    return (x - IMAGENET_MEAN) / IMAGENET_STD

rgb_all = flux_to_rgb_batch(flux)          # cache the RGB versions of the whole sample
print("RGB sample:", rgb_all.shape, rgb_all.dtype, f"range [{rgb_all.min():.2f}, {rgb_all.max():.2f}]")

i0 = int(show_idx[-1])
x0 = preprocess(rgb_all[i0:i0 + 1])
print("Model input:", tuple(x0.shape), "| mean per channel:", x0.mean((0, 2, 3)).numpy().round(2))

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
ax[0].imshow(asinh_band(flux[i0, 1]), cmap="gray"); ax[0].set_title("r band (160×160)")
ax[1].imshow(rgb_all[i0]); ax[1].set_title("g,r,z composite (160×160)")
den = (x0 * IMAGENET_STD + IMAGENET_MEAN)[0].permute(1, 2, 0).numpy()
ax[2].imshow(np.clip(den, 0, 1)); ax[2].set_title(f"model input ({CFG['img_size']}×{CFG['img_size']})")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

# How much of each image is sky? Useful context for attention / mask results later.
print(f"Mean fraction of near-black pixels in the composites: {(rgb_all.max(-1) < 0.05).mean():.1%}")

## 7. Run inference

`model.forward_features(x)` returns a dict with the normalised CLS token, register tokens (if any)
and patch tokens. `model(x)` alone returns only the CLS embedding.

In [ ]:
@torch.inference_mode()
def run_dino(x):
    x = x.to(device)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
        out = model.forward_features(x)
    return {k: (v.float().cpu() if torch.is_tensor(v) else v) for k, v in out.items()}

t0 = time.time()
out0 = run_dino(x0)
print(f"single image: {1000*(time.time()-t0):.0f} ms (first call includes warm-up)\n")
for k, v in out0.items():
    print(f"  {k:22s}", tuple(v.shape) if torch.is_tensor(v) else v)

# Sanity: model(x) == normalised CLS token
with torch.inference_mode():
    direct = model(x0.to(device)).float().cpu()
print("\nmodel(x) matches x_norm_clstoken:", torch.allclose(direct, out0["x_norm_clstoken"], atol=1e-2))

# A small batch, timed
xb = preprocess(rgb_all[:CFG["batch_size"]])
if device.type == "cuda": torch.cuda.synchronize()
t0 = time.time(); outb = run_dino(xb)
if device.type == "cuda": torch.cuda.synchronize()
dt = time.time() - t0
print(f"batch of {len(xb)}: {dt:.2f}s  -> ~{len(xb)/dt:.0f} img/s"
      f"  (full 90k dataset would take ~{90580/(len(xb)/dt)/60:.0f} min at this rate)")

## 8. Extract CLS token

The CLS token is a single vector summarising the whole image — the natural input for classification,
similarity search and t-SNE/PCA.

In [ ]:
cls0 = out0["x_norm_clstoken"][0]
print("CLS token shape:", tuple(cls0.shape))
print(f"norm={cls0.norm():.2f}  mean={cls0.mean():.3f}  std={cls0.std():.3f}")

cls_b = outb["x_norm_clstoken"]
cls_n = F.normalize(cls_b, dim=1)
sim = cls_n @ cls_n.T

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(cls0.numpy(), lw=0.6); ax[0].set_title("CLS embedding of one galaxy"); ax[0].set_xlabel("dimension")
im = ax[1].imshow(sim.numpy(), cmap="magma", origin="upper"); plt.colorbar(im, ax=ax[1])
ax[1].set_title(f"cosine similarity between {len(cls_b)} galaxies")
plt.tight_layout(); plt.show()
off = sim[~torch.eye(len(sim), dtype=bool)]
print(f"Off-diagonal cosine similarity: mean={off.mean():.3f}, min={off.min():.3f}, max={off.max():.3f}")
print("A high mean says all galaxies look 'alike' to an ImageNet-trained model — "
      "keep this number to compare against domain-adapted models later.")

## 9. Extract patch tokens

Patch tokens are one vector per 14×14 input patch — the raw material for segmentation,
part discovery and attention analysis. We reshape the token sequence back into a spatial grid.

In [ ]:
def patch_grid(out, img_size=CFG["img_size"]):
    # (B, N_patches, D) -> (B, h, w, D)
    pt = out["x_norm_patchtokens"]
    h = w = img_size // PATCH
    assert pt.shape[1] == h * w, (pt.shape, h, w)
    return pt.reshape(pt.shape[0], h, w, pt.shape[-1])

pg0 = patch_grid(out0)
print("patch tokens:", tuple(out0["x_norm_patchtokens"].shape), "-> grid", tuple(pg0.shape))
h = w = pg0.shape[1]
print(f"each token sees {PATCH}×{PATCH} px of the {CFG['img_size']}px input "
      f"= {PATCH*160/CFG['img_size']:.1f}×{PATCH*160/CFG['img_size']:.1f} px of the original 160px cutout "
      f"≈ {PATCH*160/CFG['img_size']*0.262:.2f}″ on the sky")

# Similarity of every patch to the centre patch — does the model "group" the galaxy?
c = pg0[0, h // 2, w // 2]
sim_map = F.cosine_similarity(pg0[0], c[None, None, :], dim=-1)
fig, ax = plt.subplots(1, 2, figsize=(7, 3.3))
ax[0].imshow(rgb_all[i0]); ax[0].set_title("input"); ax[0].axis("off")
im = ax[1].imshow(sim_map.numpy(), cmap="viridis"); plt.colorbar(im, ax=ax[1])
ax[1].set_title("cosine sim. to centre patch"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 10. Inspect tensor dimensions

The token count grows with the square of the input size, while attention cost grows with its square
(∝ tokens²). The 160px cutouts force a trade-off between spatial detail and compute — this table is
the starting point for the report's *resolution constraints* section.

In [ ]:
rows = []
for size in [112, 154, 168, 224, 280, 336]:
    x = preprocess(rgb_all[:8], size=size)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.time(); o = run_dino(x)
    if device.type == "cuda": torch.cuda.synchronize()
    g = size // PATCH
    rows.append(dict(input_px=size, upsample=f"{size/160:.2f}×", grid=f"{g}×{g}",
                     n_patch_tokens=o["x_norm_patchtokens"].shape[1],
                     seq_len=1 + N_REG + g * g,
                     orig_px_per_token=round(PATCH * 160 / size, 1),
                     cls_shape=str(tuple(o["x_norm_clstoken"].shape)),
                     patch_shape=str(tuple(o["x_norm_patchtokens"].shape)),
                     ms_per_8_imgs=round(1000 * (time.time() - t0))))
dims = pd.DataFrame(rows)
display(dims)

print("\nFull output of forward_features at the configured size:")
for k, v in out0.items():
    if torch.is_tensor(v):
        print(f"  {k:22s} {str(tuple(v.shape)):18s} {v.dtype}")
print(f"\nx_prenorm = [CLS | {N_REG} registers | {h*w} patches] -> length {1 + N_REG + h*w}")

## 11. Visualize patch grid

Four views of the same galaxies:

1. **Grid overlay** — exactly which pixels each token covers.
2. **Patch-token norm** — plain DINOv2 has a few very high-norm "artifact" tokens, usually on empty background.
   On mostly-empty sky cutouts these are worth spotting early.
3. **Patch PCA → RGB** (the DINOv2 paper's visualisation): PCA on patch tokens from several images, threshold the
   first component to separate foreground from sky, then map 3 components of the foreground to colours.
   Similar colours = similar features = candidate *parts*.
4. **Preview of Week 2:** last-layer CLS→patch attention for each head.

In [ ]:
def draw_grid(ax, img, n):
    H = img.shape[0]
    ax.imshow(img)
    step = H / n
    for k in range(n + 1):
        ax.axhline(k * step - 0.5, color="w", lw=0.4, alpha=0.6)
        ax.axvline(k * step - 0.5, color="w", lw=0.4, alpha=0.6)
    ax.add_patch(mpatches.Rectangle((step * (n // 2) - 0.5, step * (n // 2) - 0.5), step, step,
                                    fill=False, ec="red", lw=1.2))
    ax.set_xticks([]); ax.set_yticks([])

vis_idx = list(show_idx)
xv = preprocess(rgb_all[vis_idx])
ov = run_dino(xv)
pgv = patch_grid(ov)                                   # (B, h, w, D)
# Use the tokens *before* the final LayerNorm: artifact tokens show up as outliers in raw norm.
prenorm_patches = ov["x_prenorm"][:, 1 + N_REG:]       # drop CLS + registers
norms = prenorm_patches.norm(dim=-1).reshape(len(vis_idx), h, w)   # (B, h, w)

fig, axes = plt.subplots(3, len(vis_idx), figsize=(2.1 * len(vis_idx), 6.3))
for j, i in enumerate(vis_idx):
    draw_grid(axes[0, j], rgb_all[i], h)
    axes[1, j].imshow(norms[j].numpy(), cmap="inferno"); axes[1, j].axis("off")
    v = norms[j].ravel().numpy()
    axes[2, j].hist(v, bins=30 if np.ptp(v) > 1e-3 * abs(v.mean()) else 1, color="#C44E52")
    axes[2, j].set_yticks([])
axes[0, 0].set_ylabel(f"{h}×{w} grid\n(red = 1 token)")
axes[1, 0].set_title("patch-token L2 norm (pre-LayerNorm)", loc="left")
axes[2, 0].set_xlabel("norm")
plt.tight_layout(); plt.show()
zn = (norms - norms.mean()) / (norms.std() + 1e-8)
print(f"Tokens with norm > mean + 3σ: {(zn > 3).float().mean():.1%} — likely artifact tokens if they sit on sky.")

In [ ]:
from sklearn.decomposition import PCA

def patch_pca_rgb(pg, fg_thresh=None, flip_if_needed=True):
    # pg: (B, h, w, D). Returns fg mask (B,h,w), PCA-RGB (B,h,w,3), first component map (B,h,w)
    B, hh, ww, D = pg.shape
    X = pg.reshape(-1, D).numpy()
    p1 = PCA(n_components=1).fit_transform(X)[:, 0]
    # Orient so that the galaxy (image centre) is positive.
    centre = p1.reshape(B, hh, ww)[:, hh//2-1:hh//2+1, ww//2-1:ww//2+1].mean()
    if flip_if_needed and centre < 0:
        p1 = -p1
    thr = np.percentile(p1, 50) if fg_thresh is None else fg_thresh
    fg = p1 > thr
    rgb = np.zeros((X.shape[0], 3))
    if fg.sum() > 3:
        comp = PCA(n_components=3).fit_transform(X[fg])
        lo, hi = np.percentile(comp, 2, 0), np.percentile(comp, 98, 0)
        rgb[fg] = np.clip((comp - lo) / (hi - lo + 1e-8), 0, 1)
    return fg.reshape(B, hh, ww), rgb.reshape(B, hh, ww, 3), p1.reshape(B, hh, ww)

fg, prgb, p1 = patch_pca_rgb(pgv)
fig, axes = plt.subplots(4, len(vis_idx), figsize=(2.1 * len(vis_idx), 8.4))
for j, i in enumerate(vis_idx):
    axes[0, j].imshow(rgb_all[i])
    axes[1, j].imshow(p1[j], cmap="coolwarm")
    axes[2, j].imshow(fg[j], cmap="gray")
    axes[3, j].imshow(prgb[j])
for r, t in enumerate(["input", "1st PCA component", "foreground (PC1 > median)", "PCA(3) of foreground"]):
    axes[r, 0].set_ylabel(t, fontsize=8)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print("The 50th-percentile threshold is a placeholder; Otsu or a fixed value tuned on a few "
      "examples will work better. Note how coarse a 16×16 grid is for small galaxies.")

In [ ]:
# Week 2 preview: CLS -> patch attention of every head in the last block.
# We hook the qkv projection and recompute softmax(q·kᵀ/√d), which works regardless of whether the
# model internally uses xformers / fused attention.
def last_block_attention(x):
    store = {}
    blk = BLOCKS[-1]
    hook = blk.attn.qkv.register_forward_hook(lambda m, i, o: store.__setitem__("qkv", o.detach()))
    try:
        run_dino(x)
    finally:
        hook.remove()
    qkv = store["qkv"].float()
    B, Ntok, C3 = qkv.shape
    H = blk.attn.num_heads; d = C3 // 3 // H
    qkv = qkv.reshape(B, Ntok, 3, H, d).permute(2, 0, 3, 1, 4)
    q, k = qkv[0], qkv[1]
    attn = ((q * d ** -0.5) @ k.transpose(-2, -1)).softmax(-1)      # (B, H, N, N)
    return attn.cpu()

attn = last_block_attention(xv)
cls_attn = attn[:, :, 0, 1 + N_REG:].reshape(len(vis_idx), N_HEADS, h, w)   # (B, heads, h, w)

fig, axes = plt.subplots(len(vis_idx), N_HEADS + 2, figsize=(1.7 * (N_HEADS + 2), 1.8 * len(vis_idx)))
for j, i in enumerate(vis_idx):
    axes[j, 0].imshow(rgb_all[i])
    for hd in range(N_HEADS):
        axes[j, 1 + hd].imshow(cls_attn[j, hd].numpy(), cmap="inferno")
    axes[j, -1].imshow(cls_attn[j].mean(0).numpy(), cmap="inferno")
axes[0, 0].set_title("input")
for hd in range(N_HEADS): axes[0, 1 + hd].set_title(f"head {hd}")
axes[0, -1].set_title("mean")
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.suptitle("Last-layer CLS attention per head", y=1.0); plt.tight_layout(); plt.show()

## 12. Initial embedding exploration

Extract CLS embeddings (plus mean-pooled patch tokens) for the whole sample, then ask:
*does the embedding space organise galaxies by physical properties nobody told it about?*

No morphology labels are used here. We colour the projections by catalog quantities
(colour, brightness, size, ellipticity). An optional cell at the end repeats the exercise on Galaxy Zoo
(`gz10`) images with real morphology labels.

In [ ]:
@torch.inference_mode()
def embed_all(rgb, bs=CFG["batch_size"]):
    cls_list, mean_list = [], []
    for s in range(0, len(rgb), bs):
        o = run_dino(preprocess(rgb[s:s + bs]))
        cls_list.append(o["x_norm_clstoken"])
        mean_list.append(o["x_norm_patchtokens"].mean(1))
    return torch.cat(cls_list).numpy(), torch.cat(mean_list).numpy()

t0 = time.time()
emb_cls, emb_patchmean = embed_all(rgb_all)
print(f"Embedded {len(emb_cls)} galaxies in {time.time()-t0:.1f}s -> CLS {emb_cls.shape}, patch-mean {emb_patchmean.shape}")

EMB_FILE = os.path.join(CFG["cache_dir"], f"emb_{CFG['model_name']}_{CFG['img_size']}_{N}.npz")
np.savez(EMB_FILE, cls=emb_cls, patchmean=emb_patchmean, object_id=cat["object_id"].values.astype(str))
print("Saved", EMB_FILE)

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
from scipy.stats import spearmanr

X = normalize(emb_cls)                      # cosine geometry
pca = PCA(n_components=50, random_state=SEED).fit(X)
Xp = pca.transform(X)
print(f"Variance explained by first 2 PCs: {pca.explained_variance_ratio_[:2].sum():.1%}; "
      f"PCs needed for 90%: {np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.9) + 1}")

t0 = time.time()
Xt = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(Xp)
print(f"t-SNE: {time.time()-t0:.1f}s")

props = [("mag_r", "r mag (brighter ↓)"), ("g_minus_r", "g − r colour"),
         ("log_size", "log10 R_e [″]"), ("ellipticity", "ellipticity")]
fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))
for c, (col, lab) in enumerate(props):
    v = cat[col].values.astype(float)
    ok = np.isfinite(v)
    lo, hi = np.nanpercentile(v, [2, 98])
    for r, (Y, name) in enumerate([(Xp[:, :2], "PCA"), (Xt, "t-SNE")]):
        sc = axes[r, c].scatter(Y[ok, 0], Y[ok, 1], c=np.clip(v[ok], lo, hi), s=5, cmap="viridis")
        axes[r, c].set_title(f"{name} — {lab}")
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        plt.colorbar(sc, ax=axes[r, c], fraction=0.046)
plt.tight_layout(); plt.show()

# Quantify: rank correlation between the top PCs and each property
corr = {}
for col, _ in props:
    v = cat[col].values.astype(float); ok = np.isfinite(v)
    corr[col] = [spearmanr(Xp[ok, k], v[ok]).correlation for k in range(5)]
display(pd.DataFrame(corr, index=[f"PC{k+1}" for k in range(5)]).T.round(2)
        .style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))

In [ ]:
# Thumbnails at their t-SNE positions — the fastest way to see what the clusters mean.
def thumb_map(Y, imgs, n_grid=14, zoom=0.9, title=""):
    fig, ax = plt.subplots(figsize=(10, 10))
    Yn = (Y - Y.min(0)) / (Y.max(0) - Y.min(0))
    used = set()
    for k in np.random.RandomState(SEED).permutation(len(Y)):
        cell = tuple((Yn[k] * (n_grid - 1)).round().astype(int))
        if cell in used:
            continue
        used.add(cell)
        x, y = Yn[k]; s = zoom / n_grid
        ax.imshow(imgs[k], extent=(x - s/2, x + s/2, y - s/2, y + s/2))
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05); ax.axis("off"); ax.set_title(title)
    plt.show()

thumb_map(Xt, rgb_all, title=f"t-SNE of {CFG['model_name']} CLS embeddings ({N} DR10 galaxies)")

In [ ]:
# Nearest-neighbour retrieval (a first look at the 'similarity search' downstream task)
S = X @ X.T
queries = np.random.RandomState(1).choice(N, 5, replace=False)
k_nn = 6
fig, axes = plt.subplots(len(queries), k_nn + 1, figsize=(1.8 * (k_nn + 1), 1.9 * len(queries)))
for r, q in enumerate(queries):
    nn = np.argsort(-S[q])[1:k_nn + 1]
    axes[r, 0].imshow(rgb_all[q]); axes[r, 0].set_title("query", color="C3")
    for c, n_ in enumerate(nn):
        axes[r, c + 1].imshow(rgb_all[n_]); axes[r, c + 1].set_title(f"{S[q, n_]:.2f}")
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.suptitle("Query galaxy and its 6 nearest neighbours (cosine similarity of CLS)", y=1.0)
plt.tight_layout(); plt.show()

### 12b. (Optional) Galaxy Zoo labels — `MultimodalUniverse/gz10`

`gz10` holds the **17,736 Galaxy10 DECaLS galaxies** with Galaxy Zoo morphology labels (10 classes).
Its images are pre-made 256×256 RGB composites (not the 4-band flux cutouts), so treat this as a quick
preview of how well off-the-shelf DINOv2 separates morphologies. The proper 4-band + label pipeline comes
with the linear-probe work.

In [ ]:
RUN_GZ10_PREVIEW = True
N_GZ = 1500

GZ10_CLASSES = ["Disturbed", "Merging", "Round Smooth", "In-between Round Smooth", "Cigar Shaped Smooth",
                "Barred Spiral", "Unbarred Tight Spiral", "Unbarred Loose Spiral",
                "Edge-on without Bulge", "Edge-on with Bulge"]

if RUN_GZ10_PREVIEW:
    from datasets import load_dataset
    gz_cache = os.path.join(CFG["cache_dir"], f"gz10_{N_GZ}.npz")
    if os.path.exists(gz_cache):
        zz = np.load(gz_cache); gz_rgb, gz_y = zz["rgb"], zz["y"]
    else:
        gz = load_dataset("MultimodalUniverse/gz10", split="train", streaming=True)
        gz = gz.shuffle(seed=SEED, buffer_size=2000)
        gz_rgb, gz_y = [], []
        for k, ex in enumerate(gz):
            if k >= N_GZ: break
            im = ex["rgb_image"].convert("RGB").resize((160, 160))    # match the DR10 cutout size
            gz_rgb.append(np.asarray(im, np.float32) / 255.0); gz_y.append(int(ex["gz10_label"]))
        gz_rgb, gz_y = np.stack(gz_rgb), np.array(gz_y)
        np.savez(gz_cache, rgb=gz_rgb, y=gz_y)
    print("gz10 sample:", gz_rgb.shape, "| class counts:", np.bincount(gz_y, minlength=10))

    gz_cls, _ = embed_all(gz_rgb)
    Xg = normalize(gz_cls)
    Yg = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(
        PCA(50, random_state=SEED).fit_transform(Xg))

    fig, ax = plt.subplots(figsize=(8.5, 6.5))
    cmap = plt.get_cmap("tab10")
    for c in range(10):
        m = gz_y == c
        ax.scatter(Yg[m, 0], Yg[m, 1], s=6, color=cmap(c), label=f"{c}: {GZ10_CLASSES[c]} ({m.sum()})")
    ax.legend(markerscale=3, fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title("t-SNE of DINOv2 CLS embeddings, coloured by Galaxy Zoo class")
    plt.tight_layout(); plt.show()

    # Label-free sanity metric: how often is a galaxy's nearest neighbour the same class?
    Sg = Xg @ Xg.T; np.fill_diagonal(Sg, -np.inf)
    nn1 = Sg.argmax(1)
    chance = ((np.bincount(gz_y, minlength=10) / len(gz_y)) ** 2).sum()
    print(f"1-NN same-class rate: {(gz_y[nn1] == gz_y).mean():.1%}  (chance ≈ {chance:.1%})")

## Week 1 summary — fill in after running

**Data**
- Sample size / bands / pixel scale / PSF: …
- i-band coverage in the sample: …
- Preprocessing choice (bands → RGB, stretch, resize): …

**Model**
- Backbone, input size, patch grid, tokens per image: …
- Throughput on T4: … img/s

**First observations**
- Patch-token norms: artifact tokens present on sky? …
- Patch PCA: does the foreground separate cleanly? Do bulge/disk/arms get different colours? …
- Attention heads: which heads look at the galaxy, which at neighbours or background? …
- Embedding space: which catalog property correlates most with PC1/PC2? …
- gz10 1-NN same-class rate vs chance: …

**Open questions for Week 2**
- Does `dinov2_vits14_reg` remove the artifact tokens?
- Is 224 px (16×16) enough, or do we need 336 px / ViT-S/8-style finer patches for part discovery?
- Can `object_mask` serve as a reference for scoring unsupervised masks (IoU)? Check what it represents in the MMU docs.
- g,r,z composite vs. a 4-channel input: what's lost by dropping i?